# Import Library

In [1]:
!pip install -U langchain langchain-community langchain-core chromadb langchain_huggingface FlagEmbedding neo4j pyvi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.9 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 71.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s e

In [2]:
!pip install -U langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install python-dotenv


In [3]:
# ===== Python Standard Library =====
import logging
import traceback
import shutil
import os
import json
from dotenv import load_dotenv
from typing import List, Callable, Optional, Dict, Tuple, Set

# ===== Third-party Libraries =====
import numpy as np
import networkx as nx
from FlagEmbedding import FlagReranker
# ===== Hugging Face =====
from transformers import pipeline, AutoTokenizer
from sentence_transformers import CrossEncoder
from huggingface_hub import login
from huggingface_hub import InferenceClient
# ===== LangChain Core =====
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, AgentMiddleware, ModelRequest, ModelResponse, AgentState
from langchain_openai import ChatOpenAI
# ===== LangChain Integrations =====
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ===== LangGraph =====
from langgraph.checkpoint.memory import InMemorySaver

# ===== Databases =====
from neo4j import GraphDatabase
from chromadb import Client
from chromadb.config import Settings
from chromadb.api.models.Collection import Collection

# ===== Logging config (chỉ gọi 1 lần) =====
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


2026-01-03 04:36:58.927203: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767415019.111201      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767415019.162116      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767415019.600909      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767415019.600944      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767415019.600946      55 computation_placer.cc:177] computation placer alr

In [4]:
login()

# Import Resourse

In [5]:
reranker = FlagReranker('namdp-ptit/ViRanker',
                        use_fp16=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [7]:
logging.info("Khởi tạo embedding & database...")
# ===== Khởi tạo embedding model =====
embedding_function = HuggingFaceEmbeddings(
    model_name="dangvantuan/vietnamese-document-embedding",
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True}
)

CHROMA_PATH = "/kaggle/input/vectordatabase-v1-0-0/vector_db_30_10"
DEST_PATH = "/kaggle/working/vector_db_30_10"

# Nếu thư mục đích đã tồn tại thì xóa trước (tránh lỗi)
if os.path.exists(DEST_PATH):
    shutil.rmtree(DEST_PATH)

# Copy toàn bộ thư mục
shutil.copytree(CHROMA_PATH, DEST_PATH)


chunk_db = Chroma(
    persist_directory=DEST_PATH,
    collection_name="chunks_his_dest",
    embedding_function=embedding_function
)

/tmp/ipykernel_55/329169165.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

/tmp/ipykernel_55/329169165.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chunk_db = Chroma(


In [8]:
len(chunk_db)

9776

# Define Knowledge Graph

In [ ]:
class KnowledgeGraph:
    def __init__(self, uri: str, user: str, password: str, reranker: FlagReranker,
                 chunk_db: Optional[Collection] = None):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.chunk_db = chunk_db
        self.reranker = reranker
        self.query: Optional[str] = None
        self.subgraph: Optional[nx.MultiDiGraph] = None
    def close(self):
        self.driver.close()

    # --- các phương thức load_subgraph, set_query, compute_edge_weights, to_undirected, compute_max_spanning_trees không đổi ---
    def load_subgraph_by_chunk_ids(self, chunk_ids: List[str], hops: int = 1) -> nx.MultiDiGraph:
        if not isinstance(hops, int) or hops < 1:
            raise ValueError("Số hop phải là một số nguyên lớn hơn hoặc bằng 1.")
        cypher_query = f"""
        MATCH (s)-[r]->(o)
        WHERE type(r) <> 'MENTIONS' AND any(cid IN r.chunk_ids WHERE cid IN $chunk_ids)
        WITH collect(DISTINCT s) + collect(DISTINCT o) AS seedNodes
        UNWIND seedNodes AS seedNode
        MATCH path = (seedNode)-[rels*0..{hops}]-(neighbor)
        WHERE all(rel IN rels WHERE type(rel) <> 'MENTIONS')
        UNWIND relationships(path) AS final_rel
        RETURN DISTINCT
            startNode(final_rel).name AS s_name,
            endNode(final_rel).name AS o_name,
            type(final_rel) AS rel_type,
            final_rel.chunk_ids AS chunk_ids
        """

        with self.driver.session() as session:
            result = session.run(cypher_query, chunk_ids=chunk_ids)
            records = list(result)
            G = nx.MultiDiGraph()
            for record in records:
                s, o = record["s_name"], record["o_name"]
                rel = record["rel_type"]
                cids = record.get("chunk_ids") or []
                G.add_edge(s, o, key=rel, relation=rel, chunk_ids=list(set(cids)))
        self.subgraph = G
        return G

    def set_query(self, query: str):
        self.query = query

    def compute_edge_weights(self):
        if self.subgraph is None:
            raise ValueError("Subgraph chưa được tải.")
        if self.query is None:
            raise ValueError("Query chưa được set. Hãy gọi set_query() trước.")
        if self.chunk_db is None:
            raise ValueError("chunk_db chưa được cung cấp.")

        G = self.subgraph.copy()

        for u, v, k, data in G.edges(data=True, keys=True):
            chunk_ids = data.get("chunk_ids", [])

            best_sim = 0.0
            best_cid = None

            for cid in chunk_ids:
                try:
                    # Gọi đúng API của Chroma
                    res = self.chunk_db.similarity_search_with_score(
                        self.query,
                        k=1,
                        filter={"id": cid}
                    )

                    if not res:
                        continue

                    _, distance = res[0]
                    similarity = 1.0 - distance  # giống hệt Chroma

                    # logger.info(
                    #     f"[EDGE DETAIL] ({u}) -[{data.get('relation','?')}]-> ({v}) | "
                    #     f"chunk_id={cid} | distance={distance:.4f} | similarity={similarity:.4f}"
                    # )

                    if similarity > best_sim:
                        best_sim = similarity
                        best_cid = cid

                except Exception as e:
                    logger.warning(
                        f"Failed similarity computation for chunk_id={cid}: {e}"
                    )

            # Gán kết quả cuối cùng cho edge
            G[u][v][k]["weight"] = best_sim
            G[u][v][k]["support_chunk"] = best_cid

            # logger.info(
            #     f"[EDGE WEIGHT] ({u}) -[{data.get('relation','?')}]-> ({v}) | "
            #     f"best_chunk={best_cid} | final_weight={best_sim:.4f}"
            # )

        self.subgraph = G
        return G



    def to_undirected(self) -> nx.Graph:
        if self.subgraph is None:
            raise ValueError("Subgraph chưa được tải.")
        UG = nx.Graph()
        for u in self.subgraph.nodes():
            for v in self.subgraph.neighbors(u):
                if UG.has_edge(u, v): continue
                forward_weights = [d.get('weight', 0.0) for d in self.subgraph.get_edge_data(u, v, default={}).values()]
                backward_weights = [d.get('weight', 0.0) for d in self.subgraph.get_edge_data(v, u, default={}).values()]
                # print(f"forward_weights: {forward_weights}")
                # print(f"backward_weights: {backward_weights}")
                all_weights = forward_weights + backward_weights
                if all_weights:
                    max_weight = max(all_weights)
                    UG.add_edge(u, v, weight=max_weight)
                    # print(f"max_weight: {max_weight, u, v}")
        return UG

    def compute_max_spanning_trees(self) -> List[nx.Graph]:
        if self.subgraph is None:
            raise ValueError("Subgraph chưa được tải.")

        UG = self.to_undirected()
        components = list(nx.connected_components(UG))
        max_trees = []
        for i, component_nodes in enumerate(components):
            subg = UG.subgraph(component_nodes)
            max_tree = nx.maximum_spanning_tree(subg, weight="weight")
            max_trees.append(max_tree)
            for u, v, data in max_tree.edges(data=True):
                w = data.get("weight", 0.0)
                # logger.info(f"{u} -- {v}   (weight={w:.4f})")
        return max_trees

    def _get_mst_representations(self, mst: nx.Graph) -> Tuple[str, str, Set[str]]:
        if self.subgraph is None:
            raise ValueError("Subgraph chưa được tải để tra cứu thông tin cạnh.")

        triplets = []
        used_chunk_ids = set()

        # --------------------------------------------------
        # STEP 1: Lấy triplets + support_chunk cho từng edge trong MST
        # --------------------------------------------------
        for u, v, _ in mst.edges(data=True):

            forward_edges = self.subgraph.get_edge_data(u, v, default={})
            backward_edges = self.subgraph.get_edge_data(v, u, default={})

            best_edge_data = None
            max_weight = -1.0
            direction = ""

            for data in forward_edges.values():
                if data.get("weight", 0.0) > max_weight:
                    max_weight = data["weight"]
                    best_edge_data = data
                    direction = "forward"

            for data in backward_edges.values():
                if data.get("weight", 0.0) > max_weight:
                    max_weight = data["weight"]
                    best_edge_data = data
                    direction = "backward"

            if not best_edge_data:
                continue

            relation = best_edge_data.get("relation", "RELATED_TO")
            support_chunk = best_edge_data.get("support_chunk")

            if support_chunk:
                used_chunk_ids.add(support_chunk)

            if direction == "forward":
                triplets.append(f"<{u}, {relation}, {v}>")
            else:
                triplets.append(f"<{v}, {relation}, {u}>")

        triplet_representation = " | ".join(triplets)

        # --------------------------------------------------
        # STEP 2: Load nội dung các chunk đã được chọn
        # --------------------------------------------------
        chunk_contents = {}
        if used_chunk_ids and self.chunk_db:
            docs = self.chunk_db.get(ids=list(used_chunk_ids))
            if docs:
                chunk_contents = dict(zip(docs["ids"], docs.get("documents", [])))

        # --------------------------------------------------
        # STEP 3: DFS để tạo text mạch lạc
        # --------------------------------------------------
        if not mst.nodes:
            return "", "", set()

        start_node = list(mst.nodes)[0]
        if mst.edges:
            root_edge = max(mst.edges(data=True), key=lambda x: x[2].get("weight", 0))
            start_node = root_edge[0]

        visited_chunks = set()
        text_parts = []

        for u, v in nx.dfs_edges(mst, source=start_node):
            for data in self.subgraph.get_edge_data(u, v, default={}).values():
                cid = data.get("support_chunk")
                if cid and cid not in visited_chunks and cid in chunk_contents:
                    text_parts.append(chunk_contents[cid])
                    visited_chunks.add(cid)

            for data in self.subgraph.get_edge_data(v, u, default={}).values():
                cid = data.get("support_chunk")
                if cid and cid not in visited_chunks and cid in chunk_contents:
                    text_parts.append(chunk_contents[cid])
                    visited_chunks.add(cid)

        text_representation = "\n\n".join(text_parts)

        return text_representation, triplet_representation, used_chunk_ids


    def arrange_and_rank(self, msts: List[nx.Graph], top_k_chunks: int = 5) -> str:
        if self.query is None: raise ValueError("Query chưa được set.")

        mst_data = []
        # Chuẩn bị dữ liệu để đưa vào cross-encoder theo batch
        pairs_to_score = []
        for i, mst in enumerate(msts):
            text_repr, triplet_repr, chunk_ids = self._get_mst_representations(mst)
            if triplet_repr:
                mst_data.append({"text": text_repr, "triplets": triplet_repr, "chunks": chunk_ids})
                pairs_to_score.append((self.query, triplet_repr))

        if not pairs_to_score:
            return ""
        scores = self.reranker.compute_score(pairs_to_score)
        # =================================================================

        ranked_msts = []
        for i, data in enumerate(mst_data):
            score = scores[i]
            data["score"] = score
            ranked_msts.append(data)
        # Sắp xếp các MST dựa trên điểm số
        ranked_msts.sort(key=lambda x: x["score"], reverse=True)
        final_context_parts, final_chunk_ids = [], set()
        for item in ranked_msts:
            if len(final_chunk_ids) >= top_k_chunks:
                break
            if item["chunks"] - final_chunk_ids:
                 final_context_parts.append(item["text"])
                 final_chunk_ids.update(item["chunks"])

        final_context = "\n\n---\n\n".join(final_context_parts)
        return final_context

# Create Middleware

In [44]:
class RetrievalMiddleware(AgentMiddleware):
    def __init__(self, KG: KnowledgeGraph, top_k_chunks: int = 3, seeds: int = 3, rewriter=None):
        super().__init__()
        self.KG = KG
        self.top_k_chunks = top_k_chunks
        self.seeds = seeds
        self.rewriter = rewriter
    def before_model(self, request):
      message = request["messages"][-1]
      if not message or not message.content:
          return request

      query = message.content
      if not self.KG.chunk_db:
          return request

      # Vector search relevant chunks
      retrieved_docs = self.KG.chunk_db.similarity_search(query, k=self.seeds)
      chunk_ids = [doc.metadata.get("id") for doc in retrieved_docs if doc.metadata.get("id")]

      if not chunk_ids:
          return request

      #KG pipeline (giữ nguyên)
      self.KG.set_query(query)
      self.KG.load_subgraph_by_chunk_ids(chunk_ids, hops=1)
      self.KG.compute_edge_weights()
      msts = self.KG.compute_max_spanning_trees()
      context = self.KG.arrange_and_rank(msts, top_k_chunks=self.top_k_chunks)

      if not context.strip():
          return request

      #Thêm context như retrieval message
      context_msg = SystemMessage(content=f"Retrieved KG context:\n{context}")

      # Clone request và thêm context message TRƯỚC user query
      new_messages = request["messages"][:-1] + [context_msg, message]
      return {**request, "messages": new_messages}

In [32]:
class RetrievalMiddleware(AgentMiddleware):
    def __init__(
        self,
        KG: KnowledgeGraph,
        top_k_chunks: int = 3,
        seeds: int = 3,
        rewriter=None,
    ):
        super().__init__()
        self.KG = KG
        self.top_k_chunks = top_k_chunks
        self.seeds = seeds
        self.rewriter = rewriter

    def before_model(self, request):
        messages = request.get("messages", [])
        if not messages:
            return request

        # 🔒 1. Tìm HumanMessage CUỐI CÙNG
        last_human_idx = None
        for i in range(len(messages) - 1, -1, -1):
            if isinstance(messages[i], HumanMessage):
                last_human_idx = i
                break

        if last_human_idx is None:
            return request

        human_msg = messages[last_human_idx]
        if not human_msg.content:
            return request

        query = human_msg.content

        if not self.KG or not self.KG.chunk_db:
            return request

        # 🔎 2. Vector search lấy seed chunks
        retrieved_docs = self.KG.chunk_db.similarity_search(
            query, k=self.seeds
        )

        chunk_ids = [
            doc.metadata.get("id")
            for doc in retrieved_docs
            if doc.metadata and doc.metadata.get("id")
        ]

        if not chunk_ids:
            return request

        # 🧠 3. KG pipeline (GIỮ NGUYÊN LOGIC)
        self.KG.set_query(query)
        self.KG.load_subgraph_by_chunk_ids(chunk_ids, hops=1)
        self.KG.compute_edge_weights()
        msts = self.KG.compute_max_spanning_trees()
        context = self.KG.arrange_and_rank(
            msts, top_k_chunks=self.top_k_chunks
        )

        if not context or not context.strip():
            return request

        # 🧩 4. Chèn context TRƯỚC HumanMessage cuối
        context_msg = SystemMessage(
            content=f"Retrieved KG context:\n{context}"
        )

        new_messages = (
            messages[:last_human_idx]
            + [context_msg, human_msg]
            + messages[last_human_idx + 1:]
        )

        return {
            **request,
            "messages": new_messages
        }


In [5]:
class RetrievalMiddleware(AgentMiddleware):
    def __init__(self, vector_db, top_k: int = 3):
        super().__init__()
        self.vector_db = vector_db
        self.top_k = top_k

    def before_model(self, request):
        # Lấy user message cuối
        message = request["messages"][-1]
        if not message or not message.content:
            return request

        if not self.vector_db:
            return request

        query = message.content

        # Vector similarity search
        docs = self.vector_db.similarity_search(query, k=self.top_k)

        if not docs:
            return request

        # Ghép nội dung các chunk thành context
        context = "\n\n\n\n".join(doc.page_content for doc in docs)

        if not context.strip():
            return request

        # Tạo system message chứa context
        context_msg = SystemMessage(
            content=f"Retrieved context:\n{context}"
        )

        # Chèn context TRƯỚC user query
        new_messages = request["messages"][:-1] + [context_msg, message]

        return {
            **request,
            "messages": new_messages
        }


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import List, Union

class ReWriterMiddleware(AgentMiddleware):
    def __init__(self, rewriter_client):
        """
        rewriter_client: Là client của LLM (ví dụ OpenAI client hoặc DeepSeek client)
        """
        super().__init__()
        self.rewriter = rewriter_client
        self.history_window = 6  # Số lượng tin nhắn lịch sử tối đa

    def before_model(self, request):
        #Lấy danh sách tin nhắn (LangChain Message Objects)
        messages: List[Union[HumanMessage, AIMessage, SystemMessage]] = request["messages"]
        
        # Kiểm tra nhanh: Nếu list rỗng hoặc chỉ có 1 tin (chưa có lịch sử) -> Return luôn
        if not messages:
            return request
        
        last_message = messages[-1]
        
        # Chỉ rewrite nếu tin nhắn cuối là của Human
        if not isinstance(last_message, HumanMessage):
            return request

        current_query = last_message.content.strip()
        if not current_query:
            return request

        # Nếu chỉ có 1 tin nhắn (User hỏi câu đầu tiên), không có lịch sử để rewrite -> Return
        # Lưu ý: Cần đếm số lượng tin nhắn thực tế (bỏ qua SystemMessage đầu tiên nếu có)
        conversation_msgs = [m for m in messages if not isinstance(m, SystemMessage)]
        if len(conversation_msgs) <= 1:
             return request

        #Build Context từ Message Objects
        # Lấy k tin nhắn gần nhất TRƯỚC câu hỏi hiện tại
        history_msgs = messages[:-1] # Lấy tất cả trừ câu cuối
        
        # Cắt lấy window (ví dụ 6 câu gần nhất) để đỡ tốn token
        recent_history = history_msgs[-self.history_window:]
        
        context_str = ""
        for msg in recent_history:
            if isinstance(msg, HumanMessage):
                context_str += f"User: {msg.content}\n"
            elif isinstance(msg, AIMessage):
                context_str += f"Bot: {msg.content}\n"
            # SystemMessage thường không cần đưa vào ngữ cảnh rewrite

        #Build Prompt (Tối ưu cho Lịch sử/Du lịch VN)
        system_prompt = """
Bạn là trợ lý chuyên viết lại câu hỏi (Query Rewriter) cho chatbot Du lịch & Lịch sử Việt Nam.
Nhiệm vụ: Viết lại câu hỏi cuối cùng của người dùng thành câu đầy đủ ngữ nghĩa dựa trên lịch sử hội thoại.

Quy tắc:
1. Thay thế đại từ (nó, ông ấy, ở đó...) bằng tên riêng/địa danh cụ thể trong lịch sử.
2. Giữ nguyên câu hỏi nếu đã rõ nghĩa.
3. KHÔNG trả lời câu hỏi. Chỉ trả về câu hỏi đã viết lại.
4. KHÔNG thêm lời dẫn (VD: "Câu viết lại là...").

Ví dụ:
History: 
User: Vua Gia Long tên thật là gì?
Bot: Nguyễn Ánh.
Input: Ông ấy lên ngôi năm nào?
Output: Vua Gia Long lên ngôi năm nào?
"""
        
        user_input = f"""
Lịch sử hội thoại:
{context_str}

Câu hỏi hiện tại:
{current_query}

Câu hỏi viết lại:
"""

        #Call LLM (DeepSeek/OpenAI Client)
        # Lưu ý: Vì function này đang sync (def), nên ta dùng hàm sync của client.
        try:
            completion = self.rewriter.chat.completions.create(
                model="deepseek-ai/DeepSeek-V3.2",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                temperature=0,
                max_tokens=150
            )
            rewritten_text = completion.choices[0].message.content.strip()
            # print(f"DEBUG: {current_query} -> {rewritten_text}") # Bật lên để debug
            
        except Exception as e:
            print(f"Rewrite Error: {e}")
            rewritten_text = current_query # Fallback về câu gốc nếu lỗi

        #Thay thế tin nhắn cuối cùng bằng câu đã rewrite
        # Tạo một HumanMessage mới với nội dung đã sửa
        new_last_message = HumanMessage(content=rewritten_text)
        
        # Tạo list tin nhắn mới để trả về
        new_messages = messages[:-1] + [new_last_message]
        
        return {**request, "messages": new_messages}

In [ ]:
@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove old messages to keep conversation manageable."""
    messages = state["messages"]
    if len(messages) > 10:
        # remove the earliest two messages
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:10]]}
    return None


# Create Agent

In [45]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    max_new_tokens=5000,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

# gpu_llm = HuggingFacePipeline.from_model_id(
#     model_id="meta-llama/Llama-3.2-3B-Instruct",
#     task="text-generation",
#     device_map="auto",  # replace with device_map="auto" to use the accelerate library.
#     pipeline_kwargs={"max_new_tokens": 5000, "repetition_penalty": 1.03},
# )

summarizer = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3.2",
    task="text-generation",
    max_new_tokens=2048,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)
chat_model = ChatHuggingFace(llm=llm)

client = InferenceClient()

In [ ]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=1000,
    timeout=30,
    api_key = api_key
)

In [47]:
system_prompt = f"""
You are an AI assistant.
- Use **only the information provided in the CONTEXT**.
- **Synthesize, reason over, and rephrase** the information from the CONTEXT to answer the question.
- Answer **in Vietnamese**, clearly and coherently.
- Do NOT add any external knowledge beyond the CONTEXT.

If the CONTEXT does NOT contain enough information to answer the question, respond with exactly:
Không có thông tin trong dữ liệu.
"""
agent = create_agent(
    model = chat_model,
    system_prompt = system_prompt,
    # checkpointer=InMemorySaver(),
    middleware=[
        # ReWriterMiddleware(rewriter_client = client),
        RetrievalMiddleware(
            KG=KnowledgeGraph
            (
                NEO4J_URI,
                NEO4J_USER,
                NEO4J_PASS,
                reranker,
                chunk_db
            ),
            top_k_chunks=3,
            seeds = 3
        ),
#     RetrievalMiddleware(
#     vector_db=chunk_db,
#     top_k=3
# )
    # delete_old_messages()
    ],
    )

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
# config = {"configurable": {"thread_id": "2"}}
response = await agent.ainvoke({"messages": [
    HumanMessage("Tên thật của vua Trần Minh Tông là gì?")]})

Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
response

{'messages': [HumanMessage(content='Tên thật của vua Trần Minh Tông là gì?', additional_kwargs={}, response_metadata={}, id='9470518e-35db-45f7-957a-890416e413ee'),
  AIMessage(content='Tên thật của vua Trần Minh Tông là Trần Minh Tông, ông có tên húy là Trần Nhật Duật. Ông là vị vua thứ 4 của triều đại Trần, trị vì từ năm 1314 đến năm 1329.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 41, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CsOYAz1JwGPR5DWjphelx2PDH7QkI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b6e2e-c3d4-7d73-af83-71dc91459949-0', usage_metadata={'input_tokens':

In [18]:
response['messages'][0].content

'Tên thật của vua Trần Minh Tông là gì?'

# Create Agent(test Version)

In [17]:
with open("/kaggle/input/banmoi/file_b_updated.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(len(data))

109


In [ ]:
import asyncio
import json
import os
import shutil
from langchain_core.messages import HumanMessage

# CẤU HÌNH
SAVE_EVERY = 5
OUTPUT_FILE = "/kaggle/working/results-KG-llama.json"
MAX_CONCURRENCY = 5  #  FIX 1: Giới hạn số luồng (tùy chỉnh theo Rate Limit của API)

semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
lock = asyncio.Lock()
completed = 0

# Giả sử biến data gốc của bạn ở đây (List of Dicts)
# data = [...]

# ---------- RESUME & MERGE: Load kết quả cũ vào data ----------
def load_and_merge_data(raw_data):
    if not os.path.exists(OUTPUT_FILE):
        return set()

    try:
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            saved_list = json.load(f)

        # Tạo map để tra cứu nhanh
        saved_map = {item["id"]: item for item in saved_list}
        done_ids = set()

        # FIX 2: Merge kết quả cũ vào biến data hiện tại
        for item in raw_data:
            item_id = item["id"]
            if item_id in saved_map:
                saved_item = saved_map[item_id]
                # Kiểm tra xem item cũ đã có response chưa
                if "response" in saved_item and saved_item["response"]:
                    item["response"] = saved_item["response"]
                    if "retrieved_contexts" in saved_item:
                        item["retrieved_contexts"] = saved_item["retrieved_contexts"]
                    done_ids.add(item_id)

        print(f"Resume: Đã load lại {len(done_ids)} kết quả cũ.")
        return done_ids
    except json.JSONDecodeError:
        print(" File save bị lỗi format, sẽ chạy lại từ đầu.")
        return set()

# Gọi hàm này trước khi chạy main
# done_ids = load_and_merge_data(data)


# ---------- SNAPSHOT AN TOÀN ----------
async def save_data():
    #  FIX 4: Ghi vào file tạm trước để tránh hỏng file nếu crash
    temp_file = OUTPUT_FILE + ".tmp"
    try:
        # json.dump là hàm đồng bộ, chạy nó trong executor để không block event loop
        await asyncio.to_thread(write_json, temp_file, data)
        shutil.move(temp_file, OUTPUT_FILE)
        # print("💾 Saved checkpoint.")
    except Exception as e:
        print(f"❌ Error saving data: {e}")

def write_json(path, content):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2)


# ---------- WORKER ----------
async def run_one(item):
    global completed

    #  FIX 1: Dùng Semaphore để giới hạn concurrency
    async with semaphore:
        #  FIX 3: Try-Catch để không làm sập cả luồng nếu 1 item lỗi
        try:
            # Code logic gọi Agent
            response = await agent.ainvoke({
                "messages": [HumanMessage(item["user_input"])]
            })

            item["response"] = response["messages"][-1].content
            item.setdefault("retrieved_contexts", []).append(
                response["messages"][1].content
            )

            # Cơ chế đếm để save
            async with lock:
                completed += 1
                should_save = (completed % SAVE_EVERY == 0)

            if should_save:
                await save_data()
                print(f"✅ Progress: {completed} new items processed.")

        except Exception as e:
            print(f" Failed ID {item.get('id')}: {e}")
            # Tùy chọn: Ghi lỗi vào item để debug sau
            item["error"] = str(e)


# ---------- MAIN ----------
async def main():
    # Bước 0: Load và Merge dữ liệu cũ (Giả sử biến data đã được define)
    global done_ids
    done_ids = load_and_merge_data(data)

    tasks = [
        run_one(item)
        for item in data
        if item["id"] not in done_ids
    ]

    print(f"Bắt đầu chạy {len(tasks)} items còn lại (Concurrency: {MAX_CONCURRENCY})...")

    if tasks:
        await asyncio.gather(*tasks)

    await save_data()
    print("Hoàn tất toàn bộ!")

await main()

🚀 Bắt đầu chạy 50 items còn lại (Concurrency: 5)...


initial target device:   0%|          | 0/2 [00:00<?, ?it/s]2026-01-02 16:26:58.606072: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767371218.632950     380 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767371218.640886     380 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767371218.687080     380 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767371218.687109     380 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767371218.6871

❌ Failed ID g3_107: async generation is not supported with HuggingFacePipeline
❌ Failed ID g4_59: async generation is not supported with HuggingFacePipeline
❌ Failed ID g1_4: async generation is not supported with HuggingFacePipeline
❌ Failed ID g1_86: async generation is not supported with HuggingFacePipeline
❌ Failed ID g1_37: async generation is not supported with HuggingFacePipeline


Chunks: 100%|██████████| 1/1 [00:00<00:00, 11.98it/s]


❌ Failed ID g4_38: async generation is not supported with HuggingFacePipeline
❌ Failed ID g2_84: async generation is not supported with HuggingFacePipeline
❌ Failed ID g6_64: async generation is not supported with HuggingFacePipeline
❌ Failed ID g1_21: async generation is not supported with HuggingFacePipeline
❌ Failed ID g2_53: async generation is not supported with HuggingFacePipeline


Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

# Test Rewriting

In [ ]:
import json
import asyncio
from langchain_core.messages import HumanMessage, AIMessage

# Hàm hỗ trợ build input (như cũ)
def build_input_from_case(case_data):
    messages = []
    if case_data["history"]:
        turns = case_data["history"].split(" | ")
        for turn in turns:
            if turn.startswith("User: "):
                messages.append(HumanMessage(content=turn.replace("User: ", "").strip()))
            elif turn.startswith("Bot: "):
                messages.append(AIMessage(content=turn.replace("Bot: ", "").strip()))
    messages.append(HumanMessage(content=case_data["current_query"]))
    return messages

async def generate_rewrites_batch():
    # 1. Load dữ liệu test
    with open('/kaggle/input/rewriting/rewriting-test.json', 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    results_to_save = []
    
    print(f"Bắt đầu chạy rewrite cho {len(dataset)} cases...")

    for i, case in enumerate(dataset):
        print(f"[{i+1}/{len(dataset)}] Processing Case ID: {case['id']}...", end=" ")
        
        # Build input
        input_messages = build_input_from_case(case)
        config = {"configurable": {"thread_id": str(case['id'])}}
        
        try:
            # --- GỌI AGENT ---
            # Agent chạy qua Middleware -> Rewrite -> LLM trả lời
            response = agent.invoke({"messages": input_messages}, config=config)
            
            # --- TRÍCH XUẤT CÂU REWRITE ---
            # Tìm tin nhắn HumanMessage cuối cùng trong danh sách trả về
            # Vì Middleware đã sửa nội dung của nó trước khi đưa vào Model
            final_messages = response["messages"]
            
            # Lọc lấy các tin nhắn là của Human
            human_msgs = [m for m in final_messages if isinstance(m, HumanMessage)]
            
            if human_msgs:
                actual_rewrite = human_msgs[-1].content
            else:
                actual_rewrite = "Error: No HumanMessage found"

            print(f" Done. Rewrite: {actual_rewrite}")

            # Lưu vào list kết quả
            results_to_save.append({
                "id": case["id"],
                "history": case["history"],
                "original_query": case["current_query"],
                "golden_rewrite": case["golden_rewrite"],
                "actual_rewrite": actual_rewrite # Đây là cái Middleware đã làm
            })

        except Exception as e:
            print(f" Error: {e}")
            results_to_save.append({
                "id": case["id"],
                "error": str(e)
            })

    # 2. Lưu ra file trung gian
    with open('rewritten_outputs.json', 'w', encoding='utf-8') as f:
        json.dump(results_to_save, f, ensure_ascii=False, indent=2)
    
    print(f"\nĐã lưu kết quả rewrite vào file 'rewritten_outputs.json'")

# Chạy hàm (Nếu chạy trong script thường thì dùng asyncio.run(generate_rewrites_batch()))
await generate_rewrites_batch()

🚀 Bắt đầu chạy rewrite cho 25 cases...
[1/25] Processing Case ID: 1... Rewrite Error: 402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-695832c0-4f08ac1673e125782cc085ee;7fc019c6-caf2-414b-a701-6c464bb95f45)

You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
✅ Done. Rewrite: Ông ấy sinh năm bao nhiêu?
[2/25] Processing Case ID: 2... Rewrite Error: 402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-695832c3-7015cc455214199e61183597;fc74e321-f7fd-4849-95c0-dc2db1acd702)

You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
✅ Done. Rewrite: Vé cáp treo lên đỉnh Fansipan giá thế nào?
[3/25] Processing Case ID: 3... ✅ Done. Rewrite: Trước khi dời đô

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Done. Rewrite: Tìm resort sát biển 5 sao ở Đà Nẵng.
[12/25] Processing Case ID: 12... ✅ Done. Rewrite: Mùa thu Hà Nội có gì đẹp?
[13/25] Processing Case ID: 13... ✅ Done. Rewrite: Ngày xưa Dinh Độc Lập tên là gì?
[14/25] Processing Case ID: 14... ✅ Done. Rewrite: Buổi tối ở Hội An nên làm gì?
[15/25] Processing Case ID: 15... ✅ Done. Rewrite: Người dân sống dưới địa đạo Củ Chi như thế nào?
[16/25] Processing Case ID: 16... ✅ Done. Rewrite: Tháng nào biển ở Phú Quý êm nhất?
[17/25] Processing Case ID: 17... ✅ Done. Rewrite: Nhã nhạc cung đình Huế được UNESCO công nhận năm nào?
[18/25] Processing Case ID: 18... ✅ Done. Rewrite: Loại tour du thuyền Vịnh Hạ Long nào có giá dưới 3 triệu và bao gồm ăn uống?
[19/25] Processing Case ID: 19... ✅ Done. Rewrite: Vua Quang Trung mất năm bao nhiêu tuổi?
[20/25] Processing Case ID: 20... ✅ Done. Rewrite: Cho mình thông tin về đèo Mã Pí Lèng.
[21/25] Processing Case ID: 21... ✅ Done. Rewrite: Thành phố Hoa Phượng Đỏ là thành phố nào?
[22/25] Proces

In [ ]:
import json
import asyncio
from langchain_core.messages import HumanMessage, AIMessage

# --- CẤU HÌNH LẠI AGENT VỚI KEY MỚI Ở ĐÂY TRƯỚC KHI CHẠY ---
# (Đảm bảo biến 'agent' của bạn đã được khởi tạo lại với API Key mới hoạt động được)

async def repair_rewrites():
    # 1. Load file kết quả hiện tại
    try:
        with open('rewritten_outputs.json', 'r', encoding='utf-8') as f:
            current_results = json.load(f)
    except FileNotFoundError:
        print("Không tìm thấy file kết quả cũ.")
        return

    # 2. Load dataset gốc để lấy lại history nếu cần
    with open('/kaggle/input/rewriting/rewriting-test.json', 'r', encoding='utf-8') as f:
        original_dataset = json.load(f)
        # Tạo dict để tra cứu nhanh: id -> data
        dataset_map = {item["id"]: item for item in original_dataset}

    print("Đang quét các case bị lỗi...")
    
    repaired_count = 0
    
    for item in current_results:
        # Điều kiện nhận biết lỗi: 
        # 1. Có trường "error" 
        # 2. Hoặc actual_rewrite y hệt original_query (do logic fallback)
        is_failed = "error" in item or item["actual_rewrite"].strip() == item["original_query"].strip()
        
        # Riêng ID 1 và 2 của bạn chắc chắn lỗi, ta check ID luôn cho chắc
        if item["id"] in [1, 2] or is_failed:
            print(f"Đang chạy lại Case ID {item['id']}...")
            
            # Lấy data gốc để build input
            original_case = dataset_map.get(item["id"])
            if not original_case: continue

            # Build Input Messages
            messages = []
            if original_case["history"]:
                turns = original_case["history"].split(" | ")
                for turn in turns:
                    if turn.startswith("User: "):
                        messages.append(HumanMessage(content=turn.replace("User: ", "").strip()))
                    elif turn.startswith("Bot: "):
                        messages.append(AIMessage(content=turn.replace("Bot: ", "").strip()))
            messages.append(HumanMessage(content=original_case["current_query"]))

            try:
                # GỌI AGENT LẠI
                config = {"configurable": {"thread_id": str(item['id'])}}
                response =  agent.invoke({"messages": messages}, config=config)
                
                # Lấy kết quả
                human_msgs = [m for m in response["messages"] if isinstance(m, HumanMessage)]
                if human_msgs:
                    new_rewrite = human_msgs[-1].content
                    
                    # Cập nhật vào item hiện tại
                    item["actual_rewrite"] = new_rewrite
                    # Xóa trường error nếu có cũ
                    if "error" in item: del item["error"]
                    
                    print(f"  Đã sửa thành: {new_rewrite}")
                    repaired_count += 1
                else:
                    print("   Vẫn không tìm thấy HumanMessage.")

            except Exception as e:
                print(f"Vẫn lỗi: {e}")

    # 3. Lưu đè lại file
    if repaired_count > 0:
        with open('rewritten_outputs.json', 'w', encoding='utf-8') as f:
            json.dump(current_results, f, ensure_ascii=False, indent=2)
        print(f"\nĐã cập nhật xong {repaired_count} case vào 'rewritten_outputs.json'.")
    else:
        print("\nKhông có case nào được sửa.")

# Chạy hàm repair
await repair_rewrites()

🛠️ Đang quét các case bị lỗi...
🔄 Đang chạy lại Case ID 1...
   ✅ Đã sửa thành: Đại tướng Võ Nguyên Giáp sinh năm bao nhiêu?
🔄 Đang chạy lại Case ID 2...
   ✅ Đã sửa thành: Vé cáp treo lên đỉnh Fansipan ở Sapa giá thế nào?
🔄 Đang chạy lại Case ID 12...
   ✅ Đã sửa thành: Mùa thu Hà Nội có gì đẹp?

💾 Đã cập nhật xong 3 case vào 'rewritten_outputs.json'.


In [ ]:
import json
import os
import shutil
from langchain_core.messages import HumanMessage

# ================== CẤU HÌNH ==================
SAVE_EVERY = 5
OUTPUT_FILE = "/kaggle/working/results-KG-llama.json"

completed = 0

# Giả sử data đã được load sẵn
# data = [...]

# ================== TIỆN ÍCH ==================
def is_complete(item):
    """
    Item được coi là hoàn chỉnh nếu:
    - retrieved_contexts đã có nội dung
    - reference không rỗng
    """
    return (
        item.get("retrieved_contexts")
        and len(item["retrieved_contexts"]) > 0
        and item.get("reference", "") != ""
    )

def save_data(data):
    """
    Ghi file an toàn (atomic write)
    """
    temp_file = OUTPUT_FILE + ".tmp"
    with open(temp_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    shutil.move(temp_file, OUTPUT_FILE)

# ================== WORKER (SYNC) ==================
def run_one(item):
    global completed

    # Input đã đủ field → bỏ qua
    if is_complete(item):
        return

    try:
        response = agent.invoke({
            "messages": [HumanMessage(item["user_input"])]
        })

        # response sinh ra
        item["response"] = response["messages"][-1].content

        # chỉ ghi retrieved_contexts nếu đang trống
        item.setdefault("retrieved_contexts", [])
        if not item["retrieved_contexts"]:
            item["retrieved_contexts"].append(
                response["messages"][1].content
            )

        completed += 1

        if completed % SAVE_EVERY == 0:
            save_data(data)
            print(f"Progress: {completed} items processed")

    except Exception as e:
        item["error"] = str(e)
        print(f"Failed ID {item.get('id')}: {e}")

# ================== MAIN ==================
def main():
    print("Bắt đầu xử lý (input là source of truth)...")

    for item in data:
        run_one(item)

    save_data(data)
    print("Hoàn tất toàn bộ!")

# ================== RUN ==================
main()


🚀 Bắt đầu xử lý (input là source of truth)...


Chunks:   0%|          | 0/2 [02:45<?, ?it/s]


KeyboardInterrupt: 